In [6]:
import pandas as pd

# File paths
dashboard_path = "dashboard_database.csv"
locations_path = "Methanex_Locations.csv"

# Load CSVs
dashboard = pd.read_csv(dashboard_path)
locations = pd.read_csv(locations_path)

# Normalize city values (helps with spacing/case mismatches)
dashboard["city"] = dashboard["city"].astype(str).str.strip()
locations["city"] = locations["city"].astype(str).str.strip()

# ✅ Force "Remote" in dashboard to use Vancouver's location data
dashboard["city_merge"] = dashboard["city"].replace({"Remote": "Vancouver"})

# Keep only the columns needed from locations
locations_subset = (
    locations[["city", "lat", "lon", "type"]]
    .drop_duplicates(subset=["city"])
)

# Merge using the adjusted key
dashboard_enriched = dashboard.merge(
    locations_subset,
    left_on="city_merge",
    right_on="city",
    how="left",
    suffixes=("", "_loc")
)

# Clean up extra merge columns
dashboard_enriched = dashboard_enriched.drop(columns=["city_merge", "city_loc"], errors="ignore")

# Result: original dashboard city stays "Remote", but lat/lon/type come from Vancouver
print(dashboard_enriched[["city", "lat", "lon", "type"]].head())

# Optional: save
# dashboard_enriched.to_csv("dashboard_database_enriched.csv", index=False)

           city      lat         lon                 type
0  Medicine Hat  500.617  -1.107.410  Production Facility
1       Geismar  302.173    -909.995  Production Facility
2  Medicine Hat  500.617  -1.107.410  Production Facility
3  Medicine Hat  500.617  -1.107.410  Production Facility
4  Medicine Hat  500.617  -1.107.410  Production Facility


In [7]:
# show all unique cities with their lat/lon/type in the enriched dashboard (only one row per city)
unique_cities = dashboard_enriched[["city", "lat", "lon", "type"]].drop_duplicates()
print(unique_cities)

             city       lat         lon                 type
0    Medicine Hat  500.6170  -1.107.410  Production Facility
1         Geismar  302.1730    -909.995  Production Facility
5       Vancouver  492.8820  -1.231.162         Headquarters
8     Point Lisas   10.3853    -614.800  Production Facility
13         Remote  492.8820  -1.231.162         Headquarters
18   Punta Arenas -529.6410    -708.315  Production Facility
30        Motunui -389.8060   1.743.146  Production Facility
143      Brussels  507.1450      4.3854      Regional Office
184      Damietta  314.0860     317.828  Production Facility


In [8]:
# update the dasboard_database.csv with the enriched data (with lat/lon/type)
dashboard_enriched.to_csv("dashboard_database.csv", index=False)